In [ ]:
import requests
import pandas as pd
import sqlite3

In [ ]:
server = "https://environment.data.gov.uk/doc/bathing-water?_pageSize=1000&_view=bathing-water&_properties=latestProfile.countyName.name%2Cdistrict.alias%2ClatestSampleAssessment.sampleDateTime.ordinalYear%2ClatestComplianceAssessment.sampleYear.ordinalYear%2ClatestComplianceAssessment.assessmentQualifier%2ClatestComplianceAssessment.assessmentRegime&_lang=en%2Ccy%2Cnone&country=http%3A%2F%2Fdata.ordnancesurvey.co.uk%2Fid%2Fcountry%2Fengland&_query-id=XbY01NFqXbY"


In [ ]:
data = requests.get(server, headers={"Accept": "application/json, text/javascript, */*; q=0.01"})
result = data.json()['result']['items']

In [ ]:
df = pd.json_normalize(data.json()['result']['items'])
df = df.rename(columns={
    "eubwidNotation": "id",
    "name._value": "name",
    "samplingPoint.lat": "lat",
    "samplingPoint.long": "lon"
}).set_index("id")
df['alternate_name'] = df['name']
df

In [ ]:
locations = df[[
    "name", "alternate_name", "lat", "lon"
]].copy()

In [ ]:
locations

In [ ]:
db = sqlite3.connect("../dataset.sqlite3")
locations.to_sql("locations", db, if_exists="append")
